# full_train.ipynb — full training run for fMRI temporal interpolation

Notebook equivalent of `python train.py --config configs/default.yaml`. Same dataset, same model, same checkpoints — laid out cell-by-cell so you can tweak overrides at the top and watch progress inline.

Default data path is `/srv/fMRI-data` (H100 server). Change `OVERRIDES['data']['root']` below if running elsewhere.

**Artifacts** written to `checkpoints/full_train_notebook/`:
- `last.pt` — full resume state (model + optimizer + scheduler + history)
- `best.pt` — best validation checkpoint (only when a val split exists)
- `model_weights.pt` / `best_weights.pt` — inference-only weights for `main.py`
- `history.json` — per-epoch loss history

For a 1-epoch smoke test of the same pipeline, see `quick_test.ipynb`.

In [ ]:
# Make the project root importable when running from notebooks/.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset

from src.dataset import FMRIInterpolationDataset, split_by_file
from src.loss import HybridL1SSIMLoss
from src.model import UNet3D
from src.utils import deep_update, load_config, pick_device, seed_everything

# Reuse the exact helpers that train.py uses so this notebook stays in sync.
from train import (
    make_loader,
    maybe_resume,
    run_epoch,
    save_checkpoint,
    save_history,
    save_weights,
    zero_initialize_residual_head,
)

## Config

Load `configs/default.yaml`, then patch with notebook-specific overrides. Edit `OVERRIDES` to tune any field; everything else falls back to the YAML defaults.

In [ ]:
config = load_config(PROJECT_ROOT / 'configs' / 'default.yaml')

OVERRIDES = {
    'data': {
        'root': '/srv/fMRI-data',   # H100 dataset location; change for local runs.
        'norm_mode': 'zscore',
        'val_fraction': 0.1,
    },
    'train': {
        'epochs': 20,
        'batch_size': 4,            # null lets train.py auto-pick; here we pin to 4 for H100.
        'lr': 1.0e-4,
        'weight_decay': 1.0e-5,
        'num_workers': 8,
        'alpha': 0.5,
        'use_mask_loss': False,
        'residual': False,
        'save_every': 1,
        'log_every': 25,
        'seed': 0,
        'compile': True,            # torch.compile on CUDA; ignored elsewhere.
        'max_samples': None,        # set to a small int for smoke runs.
    },
    'checkpoint': {
        'dir': str(PROJECT_ROOT / 'checkpoints' / 'full_train_notebook'),
        'resume': None,             # path to a last.pt to continue a previous run.
    },
}
deep_update(config, OVERRIDES)

data_cfg  = config['data']
model_cfg = config['model']
train_cfg = config['train']
ckpt_cfg  = config['checkpoint']

print(json.dumps(config, indent=2, default=str))

In [ ]:
# Device + determinism knobs (matches train.py).
seed_everything(train_cfg.get('seed', 0))
torch.use_deterministic_algorithms(False)

device = pick_device(train_cfg.get('device'))
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
    print('cuda:', torch.cuda.get_device_name(0))
print('device:', device)

## Dataset + split

Recursively scans `data.root` for `*_bold.nii.gz`. When more than one file is found, splits by file using `data.val_fraction`; otherwise the single file becomes the train set with no validation.

In [ ]:
dataset = FMRIInterpolationDataset(
    root=data_cfg.get('root'),
    norm_mode=data_cfg.get('norm_mode', 'zscore'),
    file_list=data_cfg.get('file_list'),
)

max_samples = train_cfg.get('max_samples')
if max_samples is not None:
    train_set = Subset(dataset, list(range(min(max_samples, len(dataset)))))
    val_set = None
elif len(dataset.files) > 1:
    train_set, val_set = split_by_file(
        dataset,
        val_fraction=data_cfg.get('val_fraction', 0.1),
        seed=train_cfg.get('seed', 0),
    )
else:
    train_set, val_set = dataset, None

print(f'files={len(dataset.files)} total_samples={len(dataset)}')
print(f'train_samples={len(train_set)} val_samples={len(val_set) if val_set is not None else 0}')

In [ ]:
is_cuda = device.type == 'cuda'
batch_size = train_cfg.get('batch_size') or (4 if is_cuda else 1)
num_workers = train_cfg.get('num_workers')
if num_workers is None:
    num_workers = 8 if is_cuda else 0

train_loader = make_loader(train_set, batch_size, num_workers, device, shuffle=True)
val_loader = (
    make_loader(val_set, batch_size, num_workers, device, shuffle=False)
    if val_set is not None else None
)
print(f'batch_size={batch_size} num_workers={num_workers} train_batches={len(train_loader)}'
      f" val_batches={len(val_loader) if val_loader else 0}")

## Model, loss, optimizer, scheduler

In [ ]:
model = UNet3D(
    in_channels=model_cfg.get('in_channels', 2),
    out_channels=model_cfg.get('out_channels', 1),
    base_channels=model_cfg.get('base_channels', 32),
    depth=model_cfg.get('depth', 4),
).to(device)

residual = bool(train_cfg.get('residual', False))
if residual and not ckpt_cfg.get('resume'):
    zero_initialize_residual_head(model)
    print('residual=True: head zero-initialized')

data_range = 1.0 if data_cfg.get('norm_mode', 'zscore') == 'percentile' else 2.0
criterion = HybridL1SSIMLoss(alpha=train_cfg.get('alpha', 0.5), data_range=data_range).to(device)

lr = float(train_cfg.get('lr', 1e-4))
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=float(train_cfg.get('weight_decay', 1e-5)))
epochs = int(train_cfg.get('epochs', 20))
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

ckpt_dir = Path(ckpt_cfg.get('dir', PROJECT_ROOT / 'checkpoints' / 'full_train_notebook'))
ckpt_dir.mkdir(parents=True, exist_ok=True)

n_params = sum(p.numel() for p in model.parameters())
print(f'params={n_params/1e6:.2f}M epochs={epochs} lr={lr:g} ckpt_dir={ckpt_dir}')

In [ ]:
# Optional resume from a previous run.
start_epoch, best_val, history = maybe_resume(
    ckpt_cfg.get('resume'), model, optimizer, scheduler, device, epochs, lr,
)

# torch.compile only on CUDA, and only after maybe_resume (which touches the raw model).
if device.type == 'cuda' and train_cfg.get('compile'):
    model = torch.compile(model, mode='max-autotune')
    print('torch.compile enabled (mode=max-autotune)')

print(f'start_epoch={start_epoch} best_val={best_val} history_len={len(history)}')

## Train

Per-epoch loop mirrors `train.py`: train, optionally validate, scheduler step, write checkpoints + history. Step-level logs print live; epoch summaries are appended below.

In [ ]:
use_mask_loss = bool(train_cfg.get('use_mask_loss', False))
log_every = int(train_cfg.get('log_every', 25))
save_every = max(1, int(train_cfg.get('save_every', 1)))

if start_epoch > epochs:
    print(f'checkpoint already at epoch {start_epoch - 1}; nothing to do')
else:
    for epoch in range(start_epoch, epochs + 1):
        epoch_lr = optimizer.param_groups[0]['lr']
        train_loss = run_epoch(
            model, train_loader, criterion, device,
            optimizer=optimizer, residual=residual,
            use_mask_loss=use_mask_loss, epoch=epoch, log_every=log_every,
        )
        val_loss = None
        if val_loader is not None:
            val_loss = run_epoch(
                model, val_loader, criterion, device,
                residual=residual, use_mask_loss=use_mask_loss, epoch=epoch,
            )

        improved = val_loss is not None and val_loss < best_val
        if improved:
            best_val = val_loss

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'lr': epoch_lr,
        })
        scheduler.step()

        if (epoch % save_every == 0) or (epoch == epochs):
            save_checkpoint(
                ckpt_dir / 'last.pt',
                model=model, optimizer=optimizer, scheduler=scheduler,
                config=config, epoch=epoch,
                train_loss=train_loss, val_loss=val_loss,
                best_val=best_val, history=history,
            )
            save_weights(ckpt_dir / 'model_weights.pt', model)
            save_history(history, ckpt_dir / 'history.json')

        if improved:
            save_checkpoint(
                ckpt_dir / 'best.pt',
                model=model, optimizer=optimizer, scheduler=scheduler,
                config=config, epoch=epoch,
                train_loss=train_loss, val_loss=val_loss,
                best_val=best_val, history=history,
            )
            save_weights(ckpt_dir / 'best_weights.pt', model)

        print(
            f'epoch={epoch} train_loss={train_loss:.6f}'
            + (f' val_loss={val_loss:.6f}' if val_loss is not None else '')
            + (' (best)' if improved else '')
        )

print('training done. best_val =', best_val)

## Loss curves

In [ ]:
import matplotlib.pyplot as plt

if not history:
    print('no history to plot')
else:
    epochs_x   = [h['epoch'] for h in history]
    train_y    = [h['train_loss'] for h in history]
    val_y      = [h['val_loss'] for h in history]
    lr_y       = [h['lr'] for h in history]
    has_val    = any(v is not None for v in val_y)

    fig, (ax_loss, ax_lr) = plt.subplots(1, 2, figsize=(11, 4))
    ax_loss.plot(epochs_x, train_y, marker='o', label='train')
    if has_val:
        ax_loss.plot(epochs_x, [v if v is not None else float('nan') for v in val_y],
                     marker='s', label='val')
    ax_loss.set_xlabel('epoch'); ax_loss.set_ylabel('hybrid loss')
    ax_loss.set_title('loss'); ax_loss.legend(); ax_loss.grid(alpha=0.3)

    ax_lr.plot(epochs_x, lr_y, marker='o', color='tab:purple')
    ax_lr.set_xlabel('epoch'); ax_lr.set_ylabel('lr')
    ax_lr.set_title('learning rate (cosine)'); ax_lr.grid(alpha=0.3)

    fig.tight_layout(); plt.show()

## Held-out qualitative check

Pick one validation sample (or a training sample if no val split), run the trained model, and compare GT / naive midpoint / model prediction with a brain-masked error overlay. Mirrors the visualization in `quick_test.ipynb`.

In [ ]:
from scipy.ndimage import gaussian_filter

eval_set = val_set if val_set is not None else train_set
sample = eval_set[0]
x = sample['x'].unsqueeze(0).to(device)
y = sample['y'].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    raw = model(x)
    pred = (0.5 * (x[:, 0:1] + x[:, 1:2]) + raw) if residual else raw
    naive = 0.5 * (x[:, 0:1] + x[:, 1:2])
    l1_model = (pred - y).abs().mean().item()
    l1_naive = (naive - y).abs().mean().item()

print(f'held-out t={sample["t"]}  model L1 = {l1_model:.4f}  naive L1 = {l1_naive:.4f}')

v_gt    = y[0, 0].cpu().numpy()
v_pred  = pred[0, 0].cpu().numpy()
v_naive = naive[0, 0].cpu().numpy()

brain_mask = np.abs(v_gt) > np.percentile(np.abs(v_gt), 20)

def error_overlay(true_vol, pred_vol, mask, sigma=1.2, pct=95):
    err = np.abs(true_vol - pred_vol) * mask
    err = gaussian_filter(err, sigma=sigma)
    clip = np.percentile(err[mask], pct) if err[mask].size else (err.max() or 1.0)
    return np.clip(err / (clip or 1.0), 0.0, 1.0)

err_model = error_overlay(v_gt, v_pred,  brain_mask)
err_naive = error_overlay(v_gt, v_naive, brain_mask)

slice_idx = v_gt.shape[0] // 2
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, vol, title in zip(axes[0],
                          [v_gt, v_naive, v_pred],
                          ['Ground truth V_{t+1}', 'Naive midpoint', 'Model prediction']):
    ax.imshow(vol[slice_idx], cmap='gray'); ax.set_title(title); ax.axis('off')

axes[1, 0].axis('off')
for ax, err, title in zip(axes[1, 1:],
                          [err_naive, err_model],
                          ['|naive - GT|  (smoothed, brain-only)',
                           '|model - GT|  (smoothed, brain-only)']):
    ax.imshow(v_gt[slice_idx], cmap='gray')
    im = ax.imshow(err[slice_idx], cmap='magma',
                   alpha=err[slice_idx] * 0.85, vmin=0, vmax=1)
    ax.set_title(title); ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='rel. error')

fig.suptitle(f'axial slice {slice_idx} | held-out t={sample["t"]}  '
             f'(model L1={l1_model:.3f}, naive L1={l1_naive:.3f})')
fig.tight_layout(); plt.show()

## Summary

- Checkpoints + history: `checkpoints/full_train_notebook/`
- Best validation loss: see the print above.
- Next step — run inference over a full BOLD file with the trained weights:

```bash
python main.py \
    --weights checkpoints/full_train_notebook/best_weights.pt \
    --input  /srv/fMRI-data/<subject>/<...>_bold.nii.gz \
    --output results/full_train_notebook/<subject>_2x.nii.gz
```